In [1]:
import os
from dotenv import load_dotenv

# 환경 변수 로드
load_dotenv()

True

In [2]:
from langchain_neo4j import Neo4jGraph

# Neo4j Desktop 연결 설정
graph = Neo4jGraph(
    url=os.getenv("NEO4J_URI"),
    username=os.getenv("NEO4J_USERNAME"),
    password=os.getenv("NEO4J_PASSWORD"),
    database=os.getenv("NEO4J_DATABASE"),
    enhanced_schema=True
)

In [3]:
# 테스트 쿼리 실행 
cypher_query = """
MATCH (n) 
RETURN count(n) AS node_count
"""

graph.query(cypher_query)

[{'node_count': 282}]

---

## 2. **Knowledge Graph 구축**

#### 1) **데이터셋 준비**

| 필드명 | 설명 |
|-------|------|
| `id` | 고유 식별자 |
| `korean_name` | 한글명 |
| `english_name` | 영문명 |
| `code` | 종목코드 |
| `listing_date` | 상장일 |
| `fund_type` | 펀드형태 |
| `index_name` | 기초지수명 |
| `tracking_multiplier` | 추적배수 |
| `management_company` | 자산운용사 |
| `ap_company` | 지정참가회사(AP) |
| `total_fee` | 총보수(%) |
| `tax_type` | 과세유형 |
| `website` | 홈페이지 |
| `base_market` | 기초 시장 |
| `base_asset` | 기초 자산 |
| `basic_info` | 기본 정보 |
| `investment_notice` | 투자유의사항 |

In [4]:
# ETF 데이터 (CSV에서 로드)
import pandas as pd

# CSV 파일 로드
df = pd.read_csv("etf_data/etf_info_cleaned.csv", encoding="utf-8")

# 데이터 형태 확인
print(f"데이터 형태: {df.shape}")

# 데이터 샘플 확인
df.head(2)

데이터 형태: (200, 17)


,id,korean_name,english_name,code,listing_date,fund_type,index_name,tracking_multiplier,management_company,ap_company,total_fee,tax_type,website,base_market,base_asset,basic_info,investment_notice
0,ETF471760,TIGER AI반도체핵심공정,TIGER AI Semiconductor Core Tech,471760,2023-11-21,수익증권형,iSelect AI반도체핵심공정지수,1.0,미래에셋자산운용,미래에셋|NH|키움|하이|BNK|한국|이베스트|대신|유진|신영|DB|신한|삼성|메리츠,0.45,비과세,http://www.tigeretf.com,국내|코스피|코스닥,주식|업종섹터|업종테마,"- 이 ETF는 국내에 상장된 주식을 주된 투자대상자산으로 하며, “iSelect ...",- 이 ETF의 수익률은 보수 또는 비용 등 이 ETF의 순자산가치에 부의 영향을 ...
1,ETF490090,TIGER 미국AI빅테크10,TIGER US AI BIG TECH 10,490090,2024-08-27,수익증권형,KEDI 미국AI빅테크10 지수(PR),1.0,미래에셋자산운용,미래에셋|메리츠|키움|한국|신한,0.30,배당소득세(보유기간과세),http://www.tigeretf.com,해외|북미|미국,주식|업종섹터|업종테마,- 이 투자신탁은 KEDI(Korea Economic Daily Index)에서 발...,- 이 ETF의 수익률은 보수 또는 비용 등 이 ETF의 순자산가치에 부의 영향을 ...


#### 2) **ETF 속성에서 고유명사/카테고리 추출**

- ETF 데이터에서 엔티티 추출하는 함수

In [8]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI
from pydantic import BaseModel, Field
from typing import List, Dict, Any, Optional, Union
from tqdm import tqdm #tqdm : 진행 표시바(Progress Bar) 라이브러리
import pandas as pd

# ETF에서 고유명사와 카테고리 추출을 위한 Pydantic 모델 정의
class ETFEntities(BaseModel):
    """ETF 상품에서 추출한 고유명사와 카테고리 정보"""
    companies: Optional[List[str]] = Field(default=None, description="ETF에 포함된 회사/기업 이름")
    sectors: Optional[List[str]] = Field(default=None, description="ETF가 속한 산업 섹터")
    technologies: Optional[List[str]] = Field(default=None, description="ETF 관련 기술 키워드")
    markets: Optional[List[str]] = Field(default=None, description="ETF 관련 시장/지역")
    asset_classes: Optional[List[str]] = Field(default=None, description="ETF 자산 클래스(주식, 채권, 원자재 등)")
    investment_themes: Optional[List[str]] = Field(default=None, description="ETF 투자 테마(AI, 친환경, 메타버스 등)")


def extract_entities_from_etf(row: pd.Series) -> ETFEntities:
    """
    ETF 메타데이터에서 고유명사와 카테고리를 추출합니다.
    
    Args:
        row (pd.Series): ETF 데이터 행
    
    Returns:
        ETFEntities: 추출된 고유명사와 카테고리 목록
    """
    # 프롬프트 템플릿 정의
    prompt = ChatPromptTemplate.from_messages([
        ("system", """ETF 상품의 고유명사와 카테고리 추출 전문가입니다. 
        제공된 ETF 상품 정보에서 다음 항목들을 정확하게 리스트로 추출하세요:
        
        추출 가이드라인:
        - companies: ETF에 포함된 또는 관련된 회사/기업 이름. 
        - sectors: ETF가 속한 산업 섹터(IT, 금융, 헬스케어 등)
        - technologies: ETF 관련 기술 키워드(AI, 블록체인, 클라우드 등)
        - markets: ETF 관련 시장/지역(국내, 미국, 아시아 등)
        - asset_classes: ETF 자산 클래스(주식, 채권, 원자재 등)
        - investment_themes: ETF 투자 테마(AI, 친환경, 메타버스 등)
        
        각 항목은 개별 리스트에 담아야 하며, 명확한 증거가 있는 항목만 포함하세요.
        ETF 이름, 설명, 기초 지수 등에서 관련 정보를 파악하세요.
        항목이 없는 경우 빈 리스트를 반환하세요.
        """),
        ("human", """다음 ETF 상품에서 고유명사와 카테고리를 추출하세요:
        
        ETF ID: {id}
        한글명: {korean_name}
        영문명: {english_name}
        종목코드: {code}
        상장일: {listing_date}
        펀드형태: {fund_type}
        기초지수명: {index_name}
        기초 시장: {base_market}
        기초 자산: {base_asset}
        기본 정보: {basic_info}
        """)
    ])

    # LLM 및 체인 설정
    llm = ChatGoogleGenerativeAI(model = "gemini-2.5-flash-lite", temperature = 0)
    
    # 구조화된 출력을 위해 Pydantic 모델과 연결
    llm_with_structured_output = llm.with_structured_output(ETFEntities)
    
    # 프롬프트와 LLM을 연결하는 체인 구성
    chain = prompt | llm_with_structured_output
    
    # 엔티티 추출 시도
    try:
        # 데이터 행에서 필드 추출
        base_market_str = row['base_market'] if 'base_market' in row else ""
        base_asset_str = row['base_asset'] if 'base_asset' in row else ""
        basic_info = row['basic_info'] if 'basic_info' in row else ""
        
        # LLM을 통해 엔티티 추출
        entities = chain.invoke({
            "id": row['id'] if 'id' in row else "",
            "korean_name": row['korean_name'] if 'korean_name' in row else "",
            "english_name": row['english_name'] if 'english_name' in row else "",
            "code": row['code'] if 'code' in row else "",
            "listing_date": row['listing_date'] if 'listing_date' in row else "",
            "fund_type": row['fund_type'] if 'fund_type' in row else "",
            "index_name": row['index_name'] if 'index_name' in row else "",
            "base_market": base_market_str,
            "base_asset": base_asset_str,
            "basic_info": basic_info
        })
        
        return entities
    except Exception as e:
        # 에러 발생 시 로그 출력 및 빈 엔티티 반환
        print(f"ETF 엔티티 추출 중 오류 발생: {e}")
        return ETFEntities(
            companies=[],
            sectors=[],
            technologies=[],
            markets=[],
            asset_classes=[],
            investment_themes=[]
        )
    
# 첫 번째 ETF 행에서 엔티티 추출
first_row = df.iloc[0]
entities = extract_entities_from_etf(first_row)

entities

ETFEntities(companies=[], sectors=['반도체'], technologies=['AI', '자연어처리'], markets=['국내', '코스피', '코스닥'], asset_classes=['주식'], investment_themes=['AI반도체', '시스템 반도체'])

- 모든 ETF에서 엔티티 추출 (gemini-2.5-flash 사용)

In [9]:
import time
# ETF 데이터 중 상위 20개에 해당하는 데이터에서 엔티티 추출하는 함수 (API 콜 횟수 제한 이슈로 50개만 실험하였음)
def extract_all_entities(df):
    """
    모든 ETF에서 엔티티를 추출하고 결과를 저장합니다.
    
    Args:
        df (pd.DataFrame): ETF 데이터프레임
        
    Returns:
        dict: ETF ID를 키로, 추출된 엔티티를 값으로 하는 딕셔너리
    """
    all_entities = {}
    
    # 모든 ETF에 대해 엔티티 추출
    for idx, row in tqdm(df.iterrows()):
        etf_id = row['id']
        entities = extract_entities_from_etf(row)
        time.sleep(2)
        all_entities[etf_id] = entities
    
    return all_entities

# 모든 ETF에서 엔티티 추출
all_etf_entities = extract_all_entities(df.head(20))

20it [01:55,  5.78s/it]


In [10]:
print(f"추출된 엔티티 수: {len(all_etf_entities)}")
print(f"첫 번째 ETF의 엔티티: {all_etf_entities[list(all_etf_entities.keys())[0]]}")

추출된 엔티티 수: 20
첫 번째 ETF의 엔티티: companies=[] sectors=['반도체'] technologies=['AI', '자연어처리'] markets=['국내', '코스피', '코스닥'] asset_classes=['주식'] investment_themes=['AI반도체', '시스템 반도체']


In [11]:
# 추출한 엔티티를 별도로 저장
import pickle 
with open('etf_data/all_etf_entities.pkl', 'wb') as f:
    pickle.dump(all_etf_entities, f)

- 모든 엔티티에 대한 통계 처리

In [12]:
from pydantic import BaseModel, Field
from typing import List, Dict, Any, Optional

# ETF에서 고유명사와 카테고리 추출을 위한 Pydantic 모델 정의
# Pydantic 모델 : AI 답변의 규격(템플릿)을 미리 정해두는 것
# 장점 : 구조화된 출력 (Structured Output), 데이터 타입 검증 용이(Validation), AI를 위한 가이드라인 작성 가능 (Description)


class ETFEntities(BaseModel):
    """ETF 상품에서 추출한 고유명사와 카테고리 정보"""
    companies: Optional[List[str]] = Field(default=None, description="ETF에 포함된 회사/기업 이름")
    sectors: Optional[List[str]] = Field(default=None, description="ETF가 속한 산업 섹터")
    technologies: Optional[List[str]] = Field(default=None, description="ETF 관련 기술 키워드")
    markets: Optional[List[str]] = Field(default=None, description="ETF 관련 시장/지역")
    asset_classes: Optional[List[str]] = Field(default=None, description="ETF 자산 클래스(주식, 채권, 원자재 등)")
    investment_themes: Optional[List[str]] = Field(default=None, description="ETF 투자 테마(AI, 친환경, 메타버스 등)")

In [13]:
# 추출한 엔티티를 로드
import pickle
with open('etf_data/all_etf_entities.pkl', 'rb') as f:
    etf_entities = pickle.load(f)

print(f"로드된 엔티티 수: {len(etf_entities)}")
print(f"첫 번째 ETF의 엔티티: {etf_entities[list(etf_entities.keys())[0]]}")

로드된 엔티티 수: 20
첫 번째 ETF의 엔티티: companies=[] sectors=['반도체'] technologies=['AI', '자연어처리'] markets=['국내', '코스피', '코스닥'] asset_classes=['주식'] investment_themes=['AI반도체', '시스템 반도체']


In [15]:
# 엔티티 통계 수집 함수
def collect_entity_statistics(all_entities):
    """
    추출된 모든 엔티티에서 통계를 수집합니다.
    
    Args:
        all_entities (dict): ETF ID를 키로, 추출된 엔티티를 값으로 하는 딕셔너리
        
    Returns:
        dict: 엔티티 유형별 통계 정보
    """
    # 엔티티 유형별 통계 초기화
    stats = {
        'companies': set(),
        'sectors': set(),
        'technologies': set(),
        'markets': set(),
        'asset_classes': set(),
        'investment_themes': set()
    }
    
    # 모든 ETF의 엔티티 수집
    for etf_id, entities in all_entities.items():
        for entity_type in stats.keys():
            entity_list = getattr(entities, entity_type) or []
            stats[entity_type].update(entity_list)
    
    # set을 list로 변환
    for key in stats:
        stats[key] = sorted(list(stats[key]))
    
    return stats

# 엔티티 통계 수집
entity_stats = collect_entity_statistics(etf_entities)

# 결과 출력
for entity_type, entities in entity_stats.items():
    print(f"{entity_type}: {len(entities)}개")
    print(f"  {', '.join(entities)}")

companies: 12개
  Benchmark Investments, LLC, China Securities Index Co., Ltd, FnGuide, NASDAQ, NASDAQ, Inc., NYSE, Nasdaq, Nikkei Inc., QQQ, S&P Dow Jones Indices, 삼성그룹, 한국자산평가
sectors: 9개
  건설, 데이터 센터, 반도체, 부동산, 시스템반도체, 업종섹터, 업종테마, 인프라, 철강소재
technologies: 8개
  AI, AI Disruption, AI Innovation, LLM, Large Language Model, 데이터 센터, 시스템반도체, 자연어처리
markets: 10개
  국내, 글로벌, 미국, 북미, 아시아, 일본, 중국, 코스닥, 코스피, 해외
asset_classes: 12개
  국공채, 단기채권, 리츠, 부동산, 시장대표, 장기채권, 장외파생상품, 주식, 중기채, 채권, 파생상품, 혼합자산
investment_themes: 22개
  AAA등급채권, AI, AI 빅테크, AI반도체, TDF, 고배당, 금리, 기업그룹, 데이터 센터, 배당, 밸류, 시스템 반도체, 시스템반도체, 안정적인 수익 추구, 인프라, 자산배분, 철강소재, 초과수익, 초장기국고채, 커버드콜, 콜옵션, 특수채


In [16]:
# ETF 데이터프레임에 엔티티 정보 추가

df_with_entities = df.copy()
for etf_id, entities in etf_entities.items():
    # 해당 ETF 행 찾기
    idx = df_with_entities[df_with_entities['id'] == etf_id].index
    if len(idx) > 0:
        # 엔티티 정보 추가 - 배열을 |로 결합하여 문자열로 저장
        df_with_entities.loc[idx[0], 'extracted_companies'] = '|'.join(entities.companies) if entities.companies else ''
        df_with_entities.loc[idx[0], 'extracted_sectors'] = '|'.join(entities.sectors) if entities.sectors else ''
        df_with_entities.loc[idx[0], 'extracted_technologies'] = '|'.join(entities.technologies) if entities.technologies else ''
        df_with_entities.loc[idx[0], 'extracted_markets'] = '|'.join(entities.markets) if entities.markets else ''
        df_with_entities.loc[idx[0], 'extracted_asset_classes'] = '|'.join(entities.asset_classes) if entities.asset_classes else ''
        df_with_entities.loc[idx[0], 'extracted_investment_themes'] = '|'.join(entities.investment_themes) if entities.investment_themes else ''

# 엔티티가 추가된 ETF 데이터 확인
df_with_entities.head()

,id,korean_name,english_name,code,listing_date,fund_type,index_name,tracking_multiplier,management_company,ap_company,...,base_market,base_asset,basic_info,investment_notice,extracted_companies,extracted_sectors,extracted_technologies,extracted_markets,extracted_asset_classes,extracted_investment_themes
0,ETF471760,TIGER AI반도체핵심공정,TIGER AI Semiconductor Core Tech,471760,2023-11-21,수익증권형,iSelect AI반도체핵심공정지수,1.0,미래에셋자산운용,미래에셋|NH|키움|하이|BNK|한국|이베스트|대신|유진|신영|DB|신한|삼성|메리츠,...,국내|코스피|코스닥,주식|업종섹터|업종테마,"- 이 ETF는 국내에 상장된 주식을 주된 투자대상자산으로 하며, “iSelect ...",- 이 ETF의 수익률은 보수 또는 비용 등 이 ETF의 순자산가치에 부의 영향을 ...,,반도체,AI|자연어처리,국내|코스피|코스닥,주식,AI반도체|시스템 반도체
1,ETF490090,TIGER 미국AI빅테크10,TIGER US AI BIG TECH 10,490090,2024-08-27,수익증권형,KEDI 미국AI빅테크10 지수(PR),1.0,미래에셋자산운용,미래에셋|메리츠|키움|한국|신한,...,해외|북미|미국,주식|업종섹터|업종테마,- 이 투자신탁은 KEDI(Korea Economic Daily Index)에서 발...,- 이 ETF의 수익률은 보수 또는 비용 등 이 ETF의 순자산가치에 부의 영향을 ...,NYSE|NASDAQ,업종섹터|업종테마,AI|AI Innovation|AI Disruption|LLM|Large Langu...,해외|북미|미국,주식,AI|AI 빅테크
2,ETF139240,TIGER 200철강소재,TIGER 200 STEEL&,139240,2011-04-06,수익증권형,코스피 200 철강/소재,1.0,미래에셋자산운용,현대|미래에셋,...,국내|코스피,주식|업종섹터|철강소재,"이 ETF는 국내 주식을 주된 투자대상자산으로 하며, “코스피 200 철강소재 지수...","기초지수 수익률 추종을 위한 최적화(optimization) 과정, 해당 펀드와 관...",,철강소재,,국내|코스피,주식,철강소재
3,ETF346000,HANARO KAP초장기국고채,HANARO KAP Ultra Long-term KTB,346000,2020-01-16,수익증권형,KAP 초장기 국고채 지수 (총수익),1.0,엔에이치아문디자산운용,키움|미래에셋|유진|NH,...,국내|주식외,채권|국공채|장기,"이 ETF는 국내 채권을 주된 투자대상자산으로 하며, KAP 초장기국고채지수를 기초...",이 ETF는 운용실적에 따라 손익이 결정되는 실적배당상품으로 예금자보호법에 따라 보...,한국자산평가,,,국내,채권|국공채|장기채권,초장기국고채
4,ETF241180,TIGER 일본니케이225,TIGER NIKKEI225,241180,2016-03-31,수익증권형,Nikkei 225,1.0,미래에셋자산운용,한국|대신,...,해외|아시아|일본,주식|시장대표,"이 ETF는 일본 주식을 주된 투자대상자산으로 하며, ""니케이 225 지수"" 수익률...",이 ETF의 수익률은 보수 또는 비용 등 이 ETF의 순자산가치에 부의 영향을 미치...,Nikkei Inc.,,,해외|아시아|일본,주식|시장대표,


In [17]:
# 데이터프레임 저장
df_with_entities.to_csv("etf_data/etf_with_extracted_entities.csv", index=False, encoding='utf-8-sig')

### 2.2 KG 온톨로지 구현

#### 1) **스키마 정의**

- **노드 유형**:
  - ETF: 상장지수펀드 정보를 담는 노드 (id, 이름, 코드, 상장일 등 속성 포함)
  - AssetManager: 자산운용사 정보를 담는 노드 (이름, 관리 ETF 수 등 속성 포함)
  - Sector: 산업 섹터 정보 노드 (예: AI 프로세스칩, 시스템 반도체)
  - Technology: 기술 정보 노드 (예: AI, 자연어처리)
  - Market: 시장 정보 노드 (예: 국내, 해외, 코스피)
  - AssetClass: 자산 유형 노드 (예: 주식, 채권)
  - InvestmentTheme: 투자 테마 노드 (예: AI)

- **관계 유형**:
  - MANAGED_BY: ETF와 자산운용사 간의 관계 (ETF → AssetManager)
  - FOCUSES_ON: ETF와 섹터/기술/테마 간의 관계 (ETF → Sector/Technology/InvestmentTheme)
  - INVESTS_IN: ETF와 시장/자산유형 간의 관계 (ETF → Market/AssetClass)

In [18]:
# 데이터프레임 로드
import pandas as pd
df_with_entities = pd.read_csv("etf_data/etf_with_extracted_entities.csv", encoding='utf-8-sig')

df_with_entities.head()

,id,korean_name,english_name,code,listing_date,fund_type,index_name,tracking_multiplier,management_company,ap_company,...,base_market,base_asset,basic_info,investment_notice,extracted_companies,extracted_sectors,extracted_technologies,extracted_markets,extracted_asset_classes,extracted_investment_themes
0,ETF471760,TIGER AI반도체핵심공정,TIGER AI Semiconductor Core Tech,471760,2023-11-21,수익증권형,iSelect AI반도체핵심공정지수,1.0,미래에셋자산운용,미래에셋|NH|키움|하이|BNK|한국|이베스트|대신|유진|신영|DB|신한|삼성|메리츠,...,국내|코스피|코스닥,주식|업종섹터|업종테마,"- 이 ETF는 국내에 상장된 주식을 주된 투자대상자산으로 하며, “iSelect ...",- 이 ETF의 수익률은 보수 또는 비용 등 이 ETF의 순자산가치에 부의 영향을 ...,NaN,반도체,AI|자연어처리,국내|코스피|코스닥,주식,AI반도체|시스템 반도체
1,ETF490090,TIGER 미국AI빅테크10,TIGER US AI BIG TECH 10,490090,2024-08-27,수익증권형,KEDI 미국AI빅테크10 지수(PR),1.0,미래에셋자산운용,미래에셋|메리츠|키움|한국|신한,...,해외|북미|미국,주식|업종섹터|업종테마,- 이 투자신탁은 KEDI(Korea Economic Daily Index)에서 발...,- 이 ETF의 수익률은 보수 또는 비용 등 이 ETF의 순자산가치에 부의 영향을 ...,NYSE|NASDAQ,업종섹터|업종테마,AI|AI Innovation|AI Disruption|LLM|Large Langu...,해외|북미|미국,주식,AI|AI 빅테크
2,ETF139240,TIGER 200철강소재,TIGER 200 STEEL&,139240,2011-04-06,수익증권형,코스피 200 철강/소재,1.0,미래에셋자산운용,현대|미래에셋,...,국내|코스피,주식|업종섹터|철강소재,"이 ETF는 국내 주식을 주된 투자대상자산으로 하며, “코스피 200 철강소재 지수...","기초지수 수익률 추종을 위한 최적화(optimization) 과정, 해당 펀드와 관...",NaN,철강소재,NaN,국내|코스피,주식,철강소재
3,ETF346000,HANARO KAP초장기국고채,HANARO KAP Ultra Long-term KTB,346000,2020-01-16,수익증권형,KAP 초장기 국고채 지수 (총수익),1.0,엔에이치아문디자산운용,키움|미래에셋|유진|NH,...,국내|주식외,채권|국공채|장기,"이 ETF는 국내 채권을 주된 투자대상자산으로 하며, KAP 초장기국고채지수를 기초...",이 ETF는 운용실적에 따라 손익이 결정되는 실적배당상품으로 예금자보호법에 따라 보...,한국자산평가,NaN,NaN,국내,채권|국공채|장기채권,초장기국고채
4,ETF241180,TIGER 일본니케이225,TIGER NIKKEI225,241180,2016-03-31,수익증권형,Nikkei 225,1.0,미래에셋자산운용,한국|대신,...,해외|아시아|일본,주식|시장대표,"이 ETF는 일본 주식을 주된 투자대상자산으로 하며, ""니케이 225 지수"" 수익률...",이 ETF의 수익률은 보수 또는 비용 등 이 ETF의 순자산가치에 부의 영향을 미치...,Nikkei Inc.,NaN,NaN,해외|아시아|일본,주식|시장대표,NaN


#### 2) **제약조건 설정**

In [19]:
# Neo4j 데이터베이스에 Cypher 쿼리를 사용하여 제약조건 설정
# 제약조건은 노드의 특정 속성이 고유(UNIQUE)하도록 보장하여 데이터 중복을 방지함

constraints = [
    # ETF 노드의 id 속성이 고유하도록 제약조건 설정
    # 이를 통해 동일한 id를 가진 ETF가 중복 저장되는 것을 방지
    "CREATE CONSTRAINT IF NOT EXISTS FOR (e:ETF) REQUIRE e.id IS UNIQUE",
    
    # AssetManager 노드의 id 속성이 고유하도록 제약조건 설정
    # 동일한 id의 자산운용사가 여러 번 생성되는 것을 방지
    "CREATE CONSTRAINT IF NOT EXISTS FOR (c:AssetManager) REQUIRE c.id IS UNIQUE",
    
    # Sector 노드의 id 속성이 고유하도록 제약조건 설정
    # 동일한 id의 섹터가 중복 생성되는 것을 방지
    "CREATE CONSTRAINT IF NOT EXISTS FOR (s:Sector) REQUIRE s.id IS UNIQUE",
    
    # Technology 노드의 id 속성이 고유하도록 제약조건 설정
    # 동일한 id의 기술이 중복 생성되는 것을 방지
    "CREATE CONSTRAINT IF NOT EXISTS FOR (t:Technology) REQUIRE t.id IS UNIQUE",
    
    # Market 노드의 id 속성이 고유하도록 제약조건 설정
    # 동일한 id의 시장이 중복 생성되는 것을 방지
    "CREATE CONSTRAINT IF NOT EXISTS FOR (m:Market) REQUIRE m.id IS UNIQUE",
    
    # AssetClass 노드의 id 속성이 고유하도록 제약조건 설정
    # 동일한 id의 자산 클래스가 중복 생성되는 것을 방지
    "CREATE CONSTRAINT IF NOT EXISTS FOR (a:AssetClass) REQUIRE a.id IS UNIQUE",
    
    # InvestmentTheme 노드의 id 속성이 고유하도록 제약조건 설정
    # 동일한 id의 투자 테마가 중복 생성되는 것을 방지
    "CREATE CONSTRAINT IF NOT EXISTS FOR (i:InvestmentTheme) REQUIRE i.id IS UNIQUE",
]

# 정의된 모든 제약조건을 순회하며 Neo4j 데이터베이스에 적용
# graph.query() 메서드를 사용하여 각 Cypher 쿼리를 실행
for constraint in constraints:
    graph.query(constraint)

In [21]:
from langchain_neo4j.graphs.graph_document import GraphDocument, Node, Relationship
from langchain_core.documents import Document
from tqdm import tqdm
import pandas as pd
import numpy as np

# 중복 노드 생성을 방지하기 위한 딕셔너리 초기화
node_dict ={} #노드 ID를 키로 사용하여 생성된 노드 객체를 저장

# 노드 간 관계를 저장할 리스트 초기화
relationships = []

# ETF 데이터프레임을 순회하며 그래프 구조로 변환
for _, row in tqdm(df_with_entities.iterrows(), total=len(df_with_entities), desc="ETF 온톨로지 구축 중"):
    # ETF ID 생성
    etf_id = row['id']

    # ETF 노드 생성 (이미 존재하는지 확인하여 중복 방지)
    if etf_id not in node_dict:
        # ETF 노드 속성 설정
        etf_properties = {
            "id": etf_id,
            "name": row['korean_name'],
            "english_name": row['english_name'],
            "code": row['code'],
            "listing_date": row['listing_date'],
            "fund_type": row['fund_type'],
            "total_fee": row['total_fee'],
            "website": row['website'],
            "basic_info": row['basic_info'],
            "investment_notice": row['investment_notice'],
            "index_name": row['index_name'],
            "tracking_multiplier": row['tracking_multiplier'],
            "ap_company": row['ap_company'],
            "tax_type": row['tax_type'],
            "base_market": row['base_market'],
            "base_asset": row['base_asset'],
        }

        # ETF 노드 객체 생성
        etf_node = Node(
            id = etf_id,
            type = "ETF", # 노드 유형 지정
            properties = etf_properties
        )

        #생성된 ETF노드를 딕셔너리에 저장
        node_dict[etf_id] = etf_node
    
    # 자산운용사 노드 생성 및 연결
    if 'management_company' in row and pd.notna(row['management_company']):
        company_name = row['management_company']
        company_id = f"company-{company_name}"

        # 자산 운용사 노드가 아직 생성되지 않았다면 새로 생성
        if company_id not in node_dict:
            company_node = Node(
                id = company_id,
                type = "AssetManager",
                properties = {"name": company_name}
            )
            node_dict[company_id] = company_node
        
        # ETF와 자산운영사 간의 'MANAGED_BY' 관계 생성 (ETF-> AssetManager)
        relationships.append(
            Relationship(
                source = node_dict[etf_id],
                target = node_dict[company_id],
                type = "MANAGED_BY",
                properties = {}
            )
        )

     # 섹터 노드 생성 및 연결
    if 'extracted_sectors' in row and pd.notna(row['extracted_sectors']):
        sectors = str(row['extracted_sectors']).split('|')
        for sector in sectors:
            if not sector:
                continue
            sector_id = f"sector-{sector}"
            
            # 섹터 노드가 아직 생성되지 않았다면 새로 생성
            if sector_id not in node_dict:
                sector_node = Node(
                    id=sector_id,
                    type="Sector",
                    properties={"name": sector}
                )
                node_dict[sector_id] = sector_node
            
            # ETF와 섹터 간의 'FOCUSES_ON' 관계 생성 (ETF → Sector)
            relationships.append(
                Relationship(
                    source=node_dict[etf_id],
                    target=node_dict[sector_id],
                    type="FOCUSES_ON",
                    properties={}
                )
            )

    # 기술 노드 생성 및 연결
    if 'extracted_technologies' in row and pd.notna(row['extracted_technologies']):
        technologies = str(row['extracted_technologies']).split('|')
        for tech in technologies:
            if not tech:
                continue
            tech_id = f"tech-{tech}"
            
            # 기술 노드가 아직 생성되지 않았다면 새로 생성
            if tech_id not in node_dict:
                tech_node = Node(
                    id=tech_id,
                    type="Technology",
                    properties={"name": tech}
                )
                node_dict[tech_id] = tech_node
            
            # ETF와 기술 간의 'FOCUSES_ON' 관계 생성 (ETF → Technology)
            relationships.append(
                Relationship(
                    source=node_dict[etf_id],
                    target=node_dict[tech_id],
                    type="FOCUSES_ON",
                    properties={}
                )
            )

    # 시장 노드 생성 및 연결
    if 'extracted_markets' in row and pd.notna(row['extracted_markets']):
        markets = str(row['extracted_markets']).split('|')
        for market in markets:
            if not market:
                continue
            market_id = f"market-{market}"
            
            # 시장 노드가 아직 생성되지 않았다면 새로 생성
            if market_id not in node_dict:
                market_node = Node(
                    id=market_id,
                    type="Market",
                    properties={"name": market}
                )
                node_dict[market_id] = market_node
            
            # ETF와 시장 간의 'INVESTS_IN' 관계 생성 (ETF → Market)
            relationships.append(
                Relationship(
                    source=node_dict[etf_id],
                    target=node_dict[market_id],
                    type="INVESTS_IN",
                    properties={}
                )
            )

    # 자산 유형 노드 생성 및 연결
    if 'extracted_asset_classes' in row and pd.notna(row['extracted_asset_classes']):
        asset_classes = str(row['extracted_asset_classes']).split('|')
        for asset_class in asset_classes:
            if not asset_class:
                continue
            asset_id = f"asset-{asset_class}"
            
            # 자산 유형 노드가 아직 생성되지 않았다면 새로 생성
            if asset_id not in node_dict:
                asset_node = Node(
                    id=asset_id,
                    type="AssetClass",
                    properties={"name": asset_class}
                )
                node_dict[asset_id] = asset_node
            
            # ETF와 자산 유형 간의 'INVESTS_IN' 관계 생성 (ETF → AssetClass)
            relationships.append(
                Relationship(
                    source=node_dict[etf_id],
                    target=node_dict[asset_id],
                    type="INVESTS_IN",
                    properties={}
                )
            )

    # 투자 테마 노드 생성 및 연결
    if 'extracted_investment_themes' in row and pd.notna(row['extracted_investment_themes']):
        themes = str(row['extracted_investment_themes']).split('|')
        for theme in themes:
            if not theme:
                continue
            theme_id = f"theme-{theme}"
            
            # 투자 테마 노드가 아직 생성되지 않았다면 새로 생성
            if theme_id not in node_dict:
                theme_node = Node(
                    id=theme_id,
                    type="InvestmentTheme",
                    properties={"name": theme}
                )
                node_dict[theme_id] = theme_node
            
            # ETF와 투자 테마 간의 'FOCUSES_ON' 관계 생성 (ETF → InvestmentTheme)
            relationships.append(
                Relationship(
                    source=node_dict[etf_id],
                    target=node_dict[theme_id],
                    type="FOCUSES_ON",
                    properties={}
                )
            )

# 노드 딕셔너리에서 모든 노드 객체를 리스트 형태로 추출
nodes = list(node_dict.values())
nodes

ETF 온톨로지 구축 중: 100%|██████████| 200/200 [00:00<00:00, 5251.25it/s]


[Node(id='ETF471760', type='ETF', properties={'id': 'ETF471760', 'name': 'TIGER AI반도체핵심공정', 'english_name': 'TIGER AI Semiconductor Core Tech', 'code': 471760, 'listing_date': '2023-11-21', 'fund_type': '수익증권형', 'total_fee': 0.45, 'website': 'http://www.tigeretf.com', 'basic_info': '- 이 ETF는 국내에 상장된 주식을 주된 투자대상자산으로 하며, “iSelect AI반도체핵심공정 지수”의 수익률 추종을 목적으로 하는 ETF입니다. - “iSelect AI반도체핵심공정 지수”는 유가증권시장 및 코스닥시장 상장기업 중 AI 프로세스칩 및 시스템 반도체 산업의 구조에 따라 선정한 키워드를 기반으로 유관 기업을 분류하기 위해 자연어처리 키워드 필터링 기술을 활용하여 종목을 편입하는 지수입니다.', 'investment_notice': '- 이 ETF의 수익률은 보수 또는 비용 등 이 ETF의 순자산가치에 부의 영향을 미치는 다양한 이유로 인하여 기초지수 수익률과 괴리(추적오차)가 발생할 수 있습니다. - 개인투자자는 보유 ETF를 거래소에서 매도하는 방법으로만 현금화가 가능하므로 보유수익증권을 판매회사 또는 지정참가회사에 환매 신청할 수 없으며, 거래소의 거래 상황에 따라 동 ETF의 순자산가치와 거래가격이 다르게 형성될 수 있습니다. - 상품에 대한 자세한 내용은 집합투자업자 인터넷 홈페이지(http://www.tigeretf.com) 또는 금융감독원 전자공시시스템의 투자설명서를 통해 확인하시기 바랍니다.', 'index_name': 'iSelect AI반도체핵심공정지수', 'tracking_multiplier': 1.0, 'ap_company': '미래에셋|NH|키움|하이|BNK|한국|이베스트|대신|유진|신영|DB|신한|삼성|메리츠', 'ta

In [22]:
# GraphDocument 객체 생성
graph_doc = GraphDocument(
    nodes=nodes,
    relationships=relationships
)

In [23]:
# 생성된 GraphDocument를 Neo4j 데이터베이스에 저장
graph.add_graph_documents([graph_doc])

print(f"총 노드 수: {len(node_dict)}")
print(f"총 관계 수: {len(relationships)}")
print("ETF 온톨로지 구축 완료!")

총 노드 수: 282
총 관계 수: 322
ETF 온톨로지 구축 완료!


In [24]:
# Neo4j 데이터베이스의 현재 스키마 정보를 새로고침하여 최신 상태로 업데이트
graph.refresh_schema()
print(graph.schema)

Node properties:
- **ETF**
  - `id`: STRING Example: "ETF471760"
  - `name`: STRING Example: "TIGER AI반도체핵심공정"
  - `website`: STRING Example: "http://www.tigeretf.com"
  - `code`: INTEGER Min: 69500, Max: 499150
  - `basic_info`: STRING Example: "- 이 ETF는 국내에 상장된 주식을 주된 투자대상자산으로 하며, “iSelect AI반도"
  - `tracking_multiplier`: FLOAT Min: 1.0, Max: 2.0
  - `fund_type`: STRING Available options: ['수익증권형']
  - `english_name`: STRING Example: "TIGER AI Semiconductor Core Tech"
  - `base_asset`: STRING Example: "주식|업종섹터|업종테마"
  - `tax_type`: STRING Available options: ['비과세', '배당소득세(보유기간과세)', '배당소득세(해외주식투자전용ETF)', '배당소득세(분리과세부동산ETF)']
  - `ap_company`: STRING Example: "미래에셋|NH|키움|하이|BNK|한국|이베스트|대신|유진|신영|DB|신한|삼성|메리츠"
  - `total_fee`: FLOAT Min: 0.0099, Max: 0.99
  - `base_market`: STRING Example: "국내|코스피|코스닥"
  - `listing_date`: STRING Example: "2023-11-21"
  - `investment_notice`: STRING Example: "- 이 ETF의 수익률은 보수 또는 비용 등 이 ETF의 순자산가치에 부의 영향을 미치는 "
  - `index_name`: STRING Example: "iSelect AI

- **Cypher 쿼리 활용**: ETF 검색 예시

In [26]:
# AI 관련 ETF 검색 예시
cypher_query = """
// ETF 노드와 Technology 노드 간의 FOCUSES_ON 관계를 가진 패턴 매칭
MATCH (e:ETF)-[:FOCUSES_ON]->(t:Technology)

// 기술 이름에 'AI' 또는 '인공지능'이 포함된 노드만 필터링
WHERE t.name CONTAINS 'AI' OR t.name CONTAINS '인공지능'

// ETF의 id, 이름, 영문 이름, 코드 정보 반환
RETURN e.id, e.name, e.english_name, e.code

// 이름을 기준으로 오름차순 정렬
ORDER BY e.name
"""

#Neo4J 데이터베이스에 쿼리 실행
result = graph.query(cypher_query)

#결과 출력
for row in result:
    print(row)

{'e.id': 'ETF471760', 'e.name': 'TIGER AI반도체핵심공정', 'e.english_name': 'TIGER AI Semiconductor Core Tech', 'e.code': 471760}
{'e.id': 'ETF490090', 'e.name': 'TIGER 미국AI빅테크10', 'e.english_name': 'TIGER US AI BIG TECH 10', 'e.code': 490090}
{'e.id': 'ETF490090', 'e.name': 'TIGER 미국AI빅테크10', 'e.english_name': 'TIGER US AI BIG TECH 10', 'e.code': 490090}
{'e.id': 'ETF490090', 'e.name': 'TIGER 미국AI빅테크10', 'e.english_name': 'TIGER US AI BIG TECH 10', 'e.code': 490090}


In [ ]:
# 특정 섹터와 기술을 모두 포함하는 ETF 검색 예시
cypher_query ="""
// ETF 노드와 Sector, Technology 노드 간의 관계 매칭
// ETF가 주목하고 있는 Sector , ETF과 주목하고 있는 Technology를 찾는다.
MATCH (e:ETF)-[:FOCUSES_ON]->(s:Sector), (e)-[:FOCUSES_ON]->(t:Technology)

// 반도체 섹터와 AI 기술을 가진 ETF 필터링
WHERE s.name = '반도체' AND t.name ='AI'

// ETF 정보와 관련 섹터, 기술 반환
RETURN e.name, e.code, s.name AS sector, t.name AS technology
"""

result = graph.query(cypher_query)
for row in result:
    print(row)

{'e.name': 'TIGER AI반도체핵심공정', 'e.code': 471760, 'sector': '반도체', 'technology': 'AI'}


In [ ]:
# 특정 ETF와 유사한 ETF 추천 예시 (같은 섹터나 기술을 공유하는 ETF)
# 'TIGER AI반도체핵심공정' ETF와 공통 속성(섹터 또는 기술)을 공유하는 다른 ETF를 찾는 쿼리

cypher_query = """
// 기준 ETF와 다른 ETF 간의 공통 노드(섹터 또는 기술) 찾기
MATCH (e1:ETF {name: 'TIGER AI반도체핵심공정'})-[:INVESTS_IN|FOCUSES_ON]->(node)<-[:INVESTS_IN|FOCUSES_ON]-(e2:ETF)

// 자기 자신은 제외
WHERE e1 <> e2

// 유사 ETF 정보, 공통 속성 유형, 속성 이름 반환
RETURN e2.name AS similar_etf, e2.code AS code, 
       LABELS(node)[0] AS common_attribute, node.name AS attribute_name

// 5개 결과만 표시
LIMIT 5
"""

result = graph.query(cypher_query)
for row in result:
    print(row)

{'similar_etf': 'ACE미국반도체데일리타겟커버드콜(합성)', 'code': 480040, 'common_attribute': 'Sector', 'attribute_name': '반도체'}
{'similar_etf': 'TIGER 미국AI빅테크10', 'code': 490090, 'common_attribute': 'Technology', 'attribute_name': 'AI'}
{'similar_etf': 'TIGER 200철강소재', 'code': 139240, 'common_attribute': 'Market', 'attribute_name': '국내'}
{'similar_etf': 'HANARO KAP초장기국고채', 'code': 346000, 'common_attribute': 'Market', 'attribute_name': '국내'}
{'similar_etf': '히어로즈 CD금리액티브(합성)', 'code': 458210, 'common_attribute': 'Market', 'attribute_name': '국내'}


#### 3) **전문(Full-Text) 검색 인덱스 설정**

- 전문 검색 인덱스는 텍스트를 단어(토큰)로 분리
- ETF 노드의 korean_name, english_name 속성에 대한 전문 검색 인덱스 생성

In [29]:
# ETF 노드의 name과 english_name 속성에 대한 전문 검색 인덱스 생성
# CREATE FULLTEXT INDEX: 전문 검색(Full-Text Search)을 위한 인덱스를 생성하는 Cypher 명령어
# etf_name_fulltext: 생성할 인덱스의 이름으로, 나중에 이 이름으로 인덱스를 참조할 수 있음
# IF NOT EXISTS: 동일한 이름의 인덱스가 이미 존재하는 경우 오류 없이 건너뜀
# FOR (e:ETF): ETF 라벨을 가진 모든 노드에 대해 인덱스 적용
# ON EACH [e.name, e.english_name]: 각 ETF 노드의 korean_name과 english_name 속성을 인덱싱
etf_fulltext_index_query = """
// 전문 검색을 위한 인덱스 생성
// ETF 노드의 name과 english_name 속성에 대해 인덱싱
// 이미 존재하는 경우 오류 없이 건너뜀
CREATE FULLTEXT INDEX etf_name_fulltext IF NOT EXISTS 
FOR (e:ETF) ON EACH [e.name, e.english_name]
"""
graph.query(etf_fulltext_index_query)

[]

In [30]:
# 인덱스 생성 확인
graph.query("SHOW FULLTEXT INDEXES")

[{'id': 16,
  'name': 'etf_name_fulltext',
  'state': 'ONLINE',
  'populationPercent': 100.0,
  'type': 'FULLTEXT',
  'entityType': 'NODE',
  'labelsOrTypes': ['ETF'],
  'properties': ['name', 'english_name'],
  'indexProvider': 'fulltext-2.0',
  'owningConstraint': None,
  'lastRead': None,
  'readCount': None}]

In [ ]:
# 전문 검색 테스트: AI 관련 ETF 검색
cypher_query = """
// 전문 검색 인덱스를 사용하여 ETF 노드 검색
CALL db.index.fulltext.queryNodes("etf_name_fulltext", $search_term)

// 검색된 노드와 관련도 점수 반환
YIELD node, score

// 결과 반환: ETF ID, 이름, 영문 이름, 검색 관련도 점수
RETURN node.id AS ETF_ID, node.name AS ETF_Name, 
       node.english_name AS ETF_English_Name, score AS SearchRelevance

// 검색 관련도 점수 기준으로 내림차순 정렬
ORDER BY SearchRelevance DESC

// 상위 5개 결과만 표시
LIMIT 5
"""
results = graph.query(cypher_query, params={"search_term": "ai"})

for result in results:
    print(f"{result['ETF_Name']} ({result['ETF_English_Name']}) - 관련도: {result['SearchRelevance']}")
print()

SOL 미국AI소프트웨어 (SOL US AI Software) - 관련도: 1.3880746364593506
TIGER 글로벌AI&로보틱스 INDXX (TIGER Global AI & Robotics) - 관련도: 1.3880746364593506
TIGER 글로벌AI인프라액티브 (TIGER GLOBAL AI INFRA ACTIVE) - 관련도: 1.2651333808898926
TIGER AI반도체핵심공정 (TIGER AI Semiconductor Core Tech) - 관련도: 1.2651333808898926
TIGER 글로벌온디바이스AI (TIGER GLOBAL ON DEVICE AI ETF) - 관련도: 1.1621979475021362



In [32]:
# 전문 검색과 그래프 탐색 결합: AI 관련 ETF와 연결된 기술 찾기
cypher_query = """
// 전문 검색 인덱스를 사용하여 'AI'가 포함된 ETF 노드 검색
CALL db.index.fulltext.queryNodes("etf_name_fulltext", $search_term)
YIELD node as etf, score

// 검색된 ETF 노드에서 FOCUSES_ON 관계를 통해 연결된 Technology 노드 찾기
MATCH (etf)-[:FOCUSES_ON]->(tech:Technology)

// 결과 반환: ETF 이름, 검색 관련도 점수, 연결된 기술 목록
RETURN etf.name AS ETF_Name, score AS SearchRelevance, 
       collect(tech.name) AS RelatedTechnologies

// 검색 관련도 점수 기준으로 내림차순 정렬하고 상위 5개만 반환
ORDER BY SearchRelevance DESC
LIMIT 5
"""
results = graph.query(cypher_query, params={"search_term": "AI"})

for result in results:
    print(f"{result['ETF_Name']} (관련도: {result['SearchRelevance']})")
    if result['RelatedTechnologies']:
        print(f"  관련 기술: {', '.join(result['RelatedTechnologies'])}")
    else:
        print("  관련 기술 정보 없음")
    print()

TIGER AI반도체핵심공정 (관련도: 1.2651333808898926)
  관련 기술: AI, 자연어처리

TIGER 미국AI빅테크10 (관련도: 1.1621979475021362)
  관련 기술: AI, AI Innovation, AI Disruption, LLM, Large Language Model



In [ ]:
# 전문 검색 한글 테스트: '신재생 에너지' 관련 ETF 검색
cypher_query = """
// 전문 검색 인덱스를 사용하여 ETF 노드 검색
CALL db.index.fulltext.queryNodes("etf_name_fulltext", $search_term)

// 검색된 노드와 관련도 점수 반환
YIELD node, score

// 결과 반환: ETF ID, 이름, 영문 이름, 검색 관련도 점수
RETURN node.id AS ETF_ID, node.name AS ETF_Name, 
       node.english_name AS ETF_English_Name, score AS SearchRelevance

// 검색 관련도 점수 기준으로 내림차순 정렬
ORDER BY SearchRelevance DESC

// 상위 5개 결과만 표시
LIMIT 5
"""
results = graph.query(cypher_query, params={"search_term": "신재생 에너지"})

for result in results:
    print(f"{result['ETF_Name']} ({result['ETF_English_Name']}) - 관련도: {result['SearchRelevance']}")
    print()

# Neo4j에서 전문검색을 할 시에 토큰으로 나눠줄 때 토큰아이저가 영어기준으로 적용되어 있기 때문에
# 한글 검색이 잘 안되는 문제점이 있음

In [35]:
# 한국어를 지원하는 토크나이저를 적용하여 인덱스 생성 (기존 인덱스 삭제 후 생성)
# 먼저 기존 인덱스 삭제
cypher_query = """
DROP INDEX etf_name_fulltext IF EXISTS
"""
graph.query(cypher_query)

[]

In [38]:
# 한국어의 경우 cjk(Chinese, Japanese, Korean)가 통합된 analyzer를 지원
# 추가로 eventually_consistent 옵션을 사용하여 인덱스 성능 향상
cypher_query = """
CREATE FULLTEXT INDEX etf_name_fulltext IF NOT EXISTS
FOR (e:ETF) ON EACH [e.name, e.english_name]
OPTIONS {
  indexConfig: {
    `fulltext.analyzer`: 'cjk',
    `fulltext.eventually_consistent`: true
  }
}
"""
graph.query(cypher_query)

[]

In [39]:
# 전문 검색 한글 테스트: '신재생 에너지' 관련 ETF 검색 (cjk analyzer로 한국어 지원)
cypher_query = """
// 전문 검색 인덱스를 사용하여 ETF 노드 검색
CALL db.index.fulltext.queryNodes("etf_name_fulltext", $search_term)

// 검색된 노드와 관련도 점수 반환
YIELD node, score

// 결과 반환: ETF ID, 이름, 영문 이름, 검색 관련도 점수
RETURN node.id AS ETF_ID, node.name AS ETF_Name, 
       node.english_name AS ETF_English_Name, score AS SearchRelevance
       
// 검색 관련도 점수 기준으로 내림차순 정렬
ORDER BY SearchRelevance DESC

// 상위 5개 결과만 표시
LIMIT 5
"""
results = graph.query(cypher_query, params={"search_term": "신재생 에너지"})

for result in results:
    print(f"{result['ETF_Name']} ({result['ETF_English_Name']}) - 관련도: {result['SearchRelevance']}")
print()

KODEX K-신재생에너지액티브 (KODEX K-Renewable Energy Active) - 관련도: 5.904680252075195



---

### 2.3 **Text2cypher 이용한 ETF 추천**

#### 1) **스키마 정보 확인**

- LLM이 Cypher 쿼리를 생성하려면 그래프 데이터베이스의 스키마 정보가 필요

In [40]:
# 기본 스키마 정보 확인
graph.refresh_schema()
print(graph.schema)

Node properties:
- **ETF**
  - `id`: STRING Example: "ETF471760"
  - `name`: STRING Example: "TIGER AI반도체핵심공정"
  - `website`: STRING Example: "http://www.tigeretf.com"
  - `code`: INTEGER Min: 69500, Max: 499150
  - `basic_info`: STRING Example: "- 이 ETF는 국내에 상장된 주식을 주된 투자대상자산으로 하며, “iSelect AI반도"
  - `tracking_multiplier`: FLOAT Min: 1.0, Max: 2.0
  - `fund_type`: STRING Available options: ['수익증권형']
  - `english_name`: STRING Example: "TIGER AI Semiconductor Core Tech"
  - `base_asset`: STRING Example: "주식|업종섹터|업종테마"
  - `tax_type`: STRING Available options: ['비과세', '배당소득세(보유기간과세)', '배당소득세(해외주식투자전용ETF)', '배당소득세(분리과세부동산ETF)']
  - `ap_company`: STRING Example: "미래에셋|NH|키움|하이|BNK|한국|이베스트|대신|유진|신영|DB|신한|삼성|메리츠"
  - `total_fee`: FLOAT Min: 0.0099, Max: 0.99
  - `base_market`: STRING Example: "국내|코스피|코스닥"
  - `listing_date`: STRING Example: "2023-11-21"
  - `investment_notice`: STRING Example: "- 이 ETF의 수익률은 보수 또는 비용 등 이 ETF의 순자산가치에 부의 영향을 미치는 "
  - `index_name`: STRING Example: "iSelect AI

#### 2) **GraphCypherQAChain 설정**

- `GraphCypherQAChain`은 LangChain에서 제공하는 체인으로, 자연어 질문을 Cypher 쿼리로 변환하고 그 결과를 바탕으로 답변을 생성

- 작동 과정:
    1. 사용자의 자연어 질문 입력
    2. LLM을 사용하여 질문을 Cypher 쿼리로 변환
    3. 생성된 Cypher 쿼리를 Neo4j 데이터베이스에 실행
    4. 쿼리 결과를 LLM에 전달하여 자연어 답변 생성

- 주요 구성 요소
    - `cypher_generation_chain`: 자연어를 Cypher 쿼리로 변환하는 체인
    - `qa_chain`: 쿼리 결과를 바탕으로 답변을 생성하는 체인
    - `graph`: Neo4j 그래프 데이터베이스 연결 객체
    - `graph_schema`: 그래프 데이터베이스의 스키마 정보

In [5]:
from langchain_neo4j import GraphCypherQAChain
from langchain_google_genai import ChatGoogleGenerativeAI

# LLM 모델 설정
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)

# GraphCypherQAChain 생성
cypher_chain = GraphCypherQAChain.from_llm(
    llm=llm,
    graph=graph,  
    validate_cypher=True,  # Cypher 쿼리 유효성 검사
    return_intermediate_steps=True,  # 중간 단계 결과 반환
    allow_dangerous_requests=True,  # DB에 영향을 줄 수 있음을 인지하고 쿼리 실행을 허용
    cypher_kwargs={"timeout": 60},  # Cypher 쿼리 실행 시간 제한
    top_k=5  # 반환할 최대 결과 수
)

In [6]:
# 예시 질문 쿼리
cypher_chain.invoke({"query": "인프라에 투자하는 ETF는 무엇인가요?"})

{'query': '인프라에 투자하는 ETF는 무엇인가요?',
 'result': 'RISE 글로벌데이터센터리츠(합성)는 인프라에 투자하는 ETF입니다.',
 'intermediate_steps': [{'query': "MATCH (e:ETF)-[:FOCUSES_ON]->(s:Sector)\nWHERE s.name = '인프라'\nRETURN e.name AS ETFName"},
  {'context': [{'ETFName': 'RISE 글로벌데이터센터리츠(합성)'}]}]}

In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain_neo4j import GraphCypherQAChain
from langchain_google_genai import ChatGoogleGenerativeAI

#APOC : Awesome Procedures on Cypher의 약자로, Neo4j의 기본 쿼리 언어인 Cypher가 제공하지 못하는 복잡한 기능을 보완해 주는 '확장 라이브러리(Standard Library)'
CYPHER_GENERATION_TEMPLATE = """
당신은 질문을 Cypher로 번역하는 Neo4j 전문가입니다.
스키마: {schema}

중요사항: 
- Cypher 쿼리에서 APOC 함수를 사용하지 마세요!
- 전문 검색을 위해서는 반드시 "etf_name_fulltext" 인덱스 이름만 사용하세요
- 전문 검색의 올바른 형식은 다음과 같습니다:
  CALL db.index.fulltext.queryNodes("etf_name_fulltext", "검색어")
  YIELD node, score
- 펀드 이름이나 ETF 이름에서 키워드를 검색할 때는 항상 전문 검색을 사용하세요
- "RETURN *" 또는 "RETURN DISTINCT *"를 변수 없이 사용하지 마세요
- 항상 "RETURN node.name, node.code"와 같이 명시적인 반환 변수를 지정하세요
- 모든 MATCH 패턴에는 최소한 하나의 변수가 정의되어 있어야 합니다
- RETURN 문에서 사용하기 전에 항상 변수를 정의하세요
- 변수 이름은 소문자로 사용하고 이름 지정에 일관성을 유지하세요
- 전문 검색 쿼리는 반드시 다음과 같은 형식으로 작성하세요:
  CALL db.index.fulltext.queryNodes("etf_name_fulltext", "검색어")
  YIELD node, score
  RETURN node.name AS ETF_Name, node.english_name AS ETF_English_Name, score AS SearchRelevance

질문: {question}
"""
cypher_prompt = PromptTemplate(
    template=CYPHER_GENERATION_TEMPLATE,
    input_variables=["schema", "question"],
)

# ETF QA를 위한 응답 생성 프롬프트
QA_TEMPLATE = """
당신은 ETF 데이터베이스 분석 전문가로서 ETF 데이터에 대한 명확하고 간결한 정보를 한국어로 제공합니다.

질문: {question}
검색 결과: {context}

응답 가이드라인:
- 검색 결과에서 핵심 정보를 요약하세요
- ETF 데이터에 대한 명확하고 객관적인 개요를 제공하세요
- 전문적이고 유익한 톤을 사용하세요
- ETF 데이터의 중요한 패턴이나 추세를 강조하세요
- 컨텍스트가 부족하거나 근거가 없을 경우 정보가 부족하다고 명확히 언급하세요
- 추측이나 개인적인 해석은 피하세요
- 데이터베이스에서 관련 정보를 찾을 수 없는 경우, 정확한 정보를 제공할 수 없다고 명시하세요

응답 형식:
- 간략한 발견 요약으로 시작하세요
- 여러 ETF가 발견된 경우 간결한 개요를 제공하세요
- 가독성을 위해 글머리 기호나 짧은 단락을 사용하세요
- 상장일, 수수료율, 투자 테마, 시장 등과 같은 관련 세부 정보를 포함하세요
- 수치 데이터나 기술 용어를 쉽게 이해할 수 있는 언어로 번역하세요

응답 구조 예시:
"분석 결과: [주요 발견 사항 요약]

주요 특징:
- [첫 번째 중요한 통찰]
- [두 번째 중요한 통찰]

추가 정보: [필요한 경우 추가 설명]"
"""

qa_prompt = PromptTemplate(
    template=QA_TEMPLATE,
    input_variables=["question", "context"],
)

# LLM 모델 설정
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)

# GraphCypherQAChain 생성
cypher_chain = GraphCypherQAChain.from_llm(
    llm=llm,
    graph=graph,
    cypher_prompt=cypher_prompt,  # 커스텀 프롬프트 설정
    qa_prompt=qa_prompt,  # 커스텀 QA 프롬프트 설정
    validate_cypher=True,  # Cypher 쿼리 유효성 검사
    return_intermediate_steps=True,  # 중간 단계 결과 반환
    allow_dangerous_requests=True,  # DB에 영향을 줄 수 있음을 인지하고 쿼리 실행을 허용
    cypher_kwargs={"timeout": 60},  # Cypher 쿼리 실행 시간 제한
    top_k=5  # 반환할 최대 결과 수
)

In [13]:
# 예시 질문 쿼리
cypher_chain.invoke({"query": "글로벌 인프라에 투자하는 ETF는 무엇인가요?"})

{'query': '글로벌 인프라에 투자하는 ETF는 무엇인가요?',
 'result': "분석 결과: '글로벌 인프라'라는 광범위한 투자 테마에 대한 직접적인 ETF는 검색 결과에서 명확히 확인되지 않았습니다. 대신, 인프라의 한 분야인 '데이터센터 리츠'에 초점을 맞춘 ETF가 한 건 발견되었습니다.\n\n주요 특징:\n*   **ETF 명칭:** RISE 글로벌데이터센터리츠(합성)\n*   **ETF 코드:** 375270\n*   **투자 테마:** 이 ETF는 전 세계 데이터센터 관련 부동산 투자 신탁(REITs)에 투자합니다. 데이터센터는 디지털 경제의 핵심 인프라로, 정보 저장 및 처리에 필수적인 시설을 포함합니다.\n*   **운용 방식:** '합성'이라는 명칭에서 알 수 있듯이, 이 ETF는 파생상품을 활용하여 기초 지수의 성과를 추종하는 합성 복제 방식을 사용합니다.\n\n추가 정보:\n제공된 검색 결과는 '글로벌 인프라' 중에서도 특히 '데이터센터'라는 특정 하위 섹터에 집중하고 있습니다. 따라서 도로, 교량, 유틸리티, 에너지 시설 등 광범위한 글로벌 인프라 자산 전체에 투자하는 ETF를 찾으신다면, 현재 검색 결과만으로는 해당 정보를 제공하기 어렵습니다. 데이터베이스에서 더 넓은 범위의 글로벌 인프라 ETF에 대한 정보는 현재 검색 결과에 포함되어 있지 않습니다.",
 'intermediate_steps': [{'query': "cypher\nMATCH (etf:ETF)-[:INVESTS_IN]->(market:Market)\nMATCH (etf)-[:FOCUSES_ON]->(sector:Sector)\nWHERE market.name = '글로벌' AND sector.name = '인프라'\nRETURN etf.name AS ETF_Name, etf.code AS ETF_Code\n"},
  {'context': [{'ETF_Name': 'RISE 글로벌데이터센터리츠(합성)', 'ETF_Code': 375270}]}]}

In [14]:
# 예시 질문 쿼리 2
cypher_chain.invoke({"query": "인프라에 투자하는 ETF는 무엇인가요?"})

{'query': '인프라에 투자하는 ETF는 무엇인가요?',
 'result': "분석 결과: 인프라에 투자하는 ETF로 'RISE 글로벌데이터센터리츠(합성)' (코드: 375270) 한 종목이 확인되었습니다. 이 ETF는 특히 디지털 인프라의 핵심 요소인 글로벌 데이터센터 리츠(REITs)에 초점을 맞추고 있습니다.\n\n주요 특징:\n*   **ETF 명칭 및 코드**: RISE 글로벌데이터센터리츠(합성) (코드: 375270)\n*   **투자 테마**: 이 ETF는 전 세계 데이터센터 리츠(Real Estate Investment Trusts)에 투자하여 디지털 인프라 부문에 노출됩니다. 데이터센터는 클라우드 컴퓨팅, 인공지능, 빅데이터 등 현대 디지털 경제의 필수적인 기반 시설로서, 정보 저장 및 처리를 위한 물리적 인프라를 제공합니다. 이는 광범위한 인프라 투자 중에서도 특히 성장 잠재력이 높은 디지털 인프라 영역에 해당합니다.\n*   **운용 방식**: ETF 명칭에 포함된 '(합성)' 표기는 이 ETF가 실물 자산을 직접 편입하는 대신 파생상품(예: 스왑 계약)을 활용하여 기초지수의 수익률을 추종하는 합성 복제 방식을 사용함을 나타냅니다.\n\n추가 정보:\n제공된 검색 결과만으로는 해당 ETF의 정확한 상장일, 총 보수율, 운용사, 거래 시장, 상세 투자 전략 및 포트폴리오 구성 등 추가적인 세부 정보를 확인할 수 없습니다. 이러한 정보는 추가적인 데이터베이스 조회를 통해 파악될 수 있습니다.",
 'intermediate_steps': [{'query': "cypher\nMATCH (etf:ETF)-[:FOCUSES_ON]->(sector:Sector)\nWHERE sector.name = '인프라'\nRETURN etf.name AS ETF_Name, etf.code AS ETF_Code\n"},
  {'context': [{'ETF_Name': 'RISE 글로벌데이터센터리츠(합성)', 'ETF_Code': 375270}]}]}

In [15]:
# 예시 질문 쿼리 3
cypher_chain.invoke({"query": "인공지능에 투자하는 ETF는 무엇인가요?"})

{'query': '인공지능에 투자하는 ETF는 무엇인가요?',
 'result': "분석 결과: 인공지능(AI)에 투자하는 ETF로 'TIGER AI반도체핵심공정'과 'TIGER 미국AI빅테크10' 두 가지가 검색되었습니다. 이들은 각각 AI 산업의 핵심 분야인 반도체 하드웨어와 미국 대형 AI 기술 기업에 초점을 맞추고 있습니다.\n\n주요 특징:\n*   **TIGER AI반도체핵심공정 (TIGER AI Semiconductor Core Tech)**: 이 ETF는 인공지능 기술 구현에 필수적인 반도체 핵심 공정 분야에 집중적으로 투자하는 것으로 파악됩니다. AI 산업의 기반이 되는 하드웨어 기술 기업들에 대한 노출을 제공하여, AI 기술 발전의 근간이 되는 인프라에 투자하고자 하는 투자자에게 적합할 수 있습니다.\n*   **TIGER 미국AI빅테크10 (TIGER US AI BIG TECH 10)**: 이 ETF는 미국의 인공지능 관련 대형 기술 기업(빅테크) 10곳에 투자하는 것으로 보입니다. 이는 AI 기술을 선도하는 주요 미국 기업들의 소프트웨어, 플랫폼, 서비스 등 광범위한 AI 생태계에 대한 집중적인 투자를 목표로 할 가능성이 높습니다.\n*   두 ETF 모두 'TIGER' 시리즈로, 미래에셋자산운용에서 운용하는 상품으로 추정됩니다. 인공지능이라는 공통된 투자 테마를 가지고 있지만, 하나는 AI 반도체 하드웨어에, 다른 하나는 미국 AI 빅테크 소프트웨어/플랫폼 기업에 초점을 맞추고 있어 투자 접근 방식에 차이가 있습니다.\n\n추가 정보:\n제공된 검색 결과에는 ETF의 상장일, 수수료율, 정확한 투자 시장(국가 명시 외), 상세한 투자 테마 및 구성 종목 등 추가적인 세부 정보가 포함되어 있지 않습니다. 따라서 이러한 정보는 현재 데이터만으로는 제공하기 어렵습니다.",
 'intermediate_steps': [{'query': "cypher\nMATCH (etf:ETF)-[:FOCUSES_ON]->(tech:Technology)\nWH

#### 3) **Few-shot Prompt**

In [10]:
from langchain_core.example_selectors import SemanticSimilarityExampleSelector
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_neo4j import Neo4jVector

# ETF 관련 few-shot 예제
ETF_EXAMPLES = [
    {
        "question": "인공지능 기술에 투자하는 ETF는 무엇인가요?",
        "query": "MATCH (e:ETF)-[:FOCUSES_ON]->(t:Technology {name: '인공지능'}) RETURN e.korean_name, e.code, e.basic_info LIMIT 5",
    },
    {
        "question": "미래에셋자산운용에서 관리하는 ETF 중 해외 시장에 투자하는 것은?",
        "query": "MATCH (e:ETF)-[:MANAGED_BY]->(:AssetManager {name: '미래에셋자산운용'}), (e)-[:INVESTS_IN]->(:Market {name: '해외'}) RETURN e.korean_name, e.code, e.base_market LIMIT 5",
    },
    {
        "question": "총 보수가 0.5% 미만인 주식형 ETF는 무엇인가요?",
        "query": "MATCH (e:ETF)-[:INVESTS_IN]->(:AssetClass {name: '주식'}) WHERE e.total_fee < 0.5 RETURN e.korean_name, e.code, e.total_fee ORDER BY e.total_fee ASC LIMIT 5",
    },
    {
        "question": "신재생에너지 테마 ETF 중 상장일이 가장 최근인 것은?",
        "query": "MATCH (e:ETF)-[:FOCUSES_ON]->(:InvestmentTheme {name: '신재생에너지'}) RETURN e.korean_name, e.code, e.listing_date ORDER BY e.listing_date DESC LIMIT 5",
    },
    {
        "question": "반도체 섹터에 투자하는 ETF 중 총 보수가 가장 낮은 것은?",
        "query": "MATCH (e:ETF)-[:BELONGS_TO]->(:Sector {name: '반도체'}) RETURN e.korean_name, e.code, e.total_fee ORDER BY e.total_fee ASC LIMIT 5",
    },
    {
        "question": "레버리지(추적배수 2배) ETF 중 국내 시장에 투자하는 것은?",
        "query": "MATCH (e:ETF)-[:INVESTS_IN]->(:Market {name: '국내'}) WHERE e.tracking_multiplier = 2.0 RETURN e.korean_name, e.code, e.tracking_multiplier LIMIT 5",
    },
    {
        "question": "원자력 테마에 투자하는 ETF 중 상장일이 가장 오래된 것은?",
        "query": "MATCH (e:ETF)-[:FOCUSES_ON]->(:InvestmentTheme {name: '원자력'}) RETURN e.korean_name, e.code, e.listing_date ORDER BY e.listing_date ASC LIMIT 5",
    },
    {
        "question": "비과세 혜택이 있는 ETF 중 해외 주식에 투자하는 것은?",
        "query": "MATCH (e:ETF)-[:INVESTS_IN]->(:AssetClass {name: '주식'}), (e)-[:INVESTS_IN]->(:Market {name: '해외'}) WHERE e.tax_type = '비과세' RETURN e.korean_name, e.code, e.tax_type LIMIT 5",
    },
    {
        "question": "교보악사자산운용에서 관리하는 ETF는 무엇인가요?",
        "query": "MATCH (e:ETF)-[:MANAGED_BY]->(:AssetManager {name: '교보악사자산운용'}) RETURN e.korean_name, e.code, e.fund_type LIMIT 5",
    },
    {
        "question": "통화 자산에 투자하는 ETF 중 총 보수가 가장 낮은 것은?",
        "query": "MATCH (e:ETF)-[:INVESTS_IN]->(:AssetClass {name: '통화'}) RETURN e.korean_name, e.code, e.total_fee ORDER BY e.total_fee ASC LIMIT 5",
    },
    {
        "question": "시스템 반도체와 AI 기술 모두에 투자하는 ETF는 무엇인가요?",
        "query": "MATCH (e:ETF)-[:BELONGS_TO]->(s:Sector {name: '시스템 반도체'}), (e)-[:FOCUSES_ON]->(t:Technology {name: 'AI'}) RETURN e.korean_name, e.code, s.name AS sector, t.name AS technology LIMIT 5",
    },
    {
        "question": "전문 검색으로 인공지능 관련 ETF를 찾고 관련도 점수를 알려주세요.",
        "query": "CALL db.index.fulltext.queryNodes('etf_name_fulltext', 'AI') YIELD node, score RETURN node.korean_name AS ETF_Name, node.english_name AS ETF_English_Name, score AS SearchRelevance ORDER BY SearchRelevance DESC LIMIT 5",
    },
    {
        "question": "AI 관련 ETF와 연결된 기술들을 모두 찾아주세요.",
        "query": "CALL db.index.fulltext.queryNodes('etf_name_fulltext', 'AI') YIELD node as etf, score MATCH (etf)-[:FOCUSES_ON]->(tech:Technology) RETURN etf.korean_name AS ETF_Name, score AS SearchRelevance, collect(tech.name) AS RelatedTechnologies ORDER BY SearchRelevance DESC LIMIT 5",
    },
    {
        "question": "인버스 ETF 중 거래량이 가장 많은 것은?",
        "query": "MATCH (e:ETF) WHERE e.tracking_multiplier < 0 RETURN e.korean_name, e.code, e.trading_volume ORDER BY e.trading_volume DESC LIMIT 5",
    },
    {
        "question": "배당수익률이 3% 이상인 ETF 중 국내 시장에 투자하는 것은?",
        "query": "MATCH (e:ETF)-[:INVESTS_IN]->(:Market {name: '국내'}) WHERE e.dividend_yield >= 3.0 RETURN e.korean_name, e.code, e.dividend_yield ORDER BY e.dividend_yield DESC LIMIT 5",
    }
] 

# 시맨틱 유사성 기반 예제 선택기 생성
example_selector = SemanticSimilarityExampleSelector.from_examples(
    ETF_EXAMPLES, 
    GoogleGenerativeAIEmbeddings(model="models/text-embedding-004"), 
    Neo4jVector, 
    k=3,  # 가장 유사한 3개의 예제 선택
    input_keys=["question"],
    url=os.environ["NEO4J_URI"],
    username=os.environ["NEO4J_USERNAME"],
    password=os.environ["NEO4J_PASSWORD"],
    database=os.environ["NEO4J_DATABASE"],  # Neo4j 데이터베이스 이름
)


# 예제 선택기 테스트
example_selector.select_examples({"question": "전문 검색으로 AI 관련 ETF를 찾아서 정리해주세요."})

[{'question': 'AI 관련 ETF와 연결된 기술들을 모두 찾아주세요.',
  'query': "CALL db.index.fulltext.queryNodes('etf_name_fulltext', 'AI') YIELD node as etf, score MATCH (etf)-[:FOCUSES_ON]->(tech:Technology) RETURN etf.korean_name AS ETF_Name, score AS SearchRelevance, collect(tech.name) AS RelatedTechnologies ORDER BY SearchRelevance DESC LIMIT 5"},
 {'question': '시스템 반도체와 AI 기술 모두에 투자하는 ETF는 무엇인가요?',
  'query': "MATCH (e:ETF)-[:BELONGS_TO]->(s:Sector {name: '시스템 반도체'}), (e)-[:FOCUSES_ON]->(t:Technology {name: 'AI'}) RETURN e.korean_name, e.code, s.name AS sector, t.name AS technology LIMIT 5"},
 {'question': '전문 검색으로 인공지능 관련 ETF를 찾고 관련도 점수를 알려주세요.',
  'query': "CALL db.index.fulltext.queryNodes('etf_name_fulltext', 'AI') YIELD node, score RETURN node.korean_name AS ETF_Name, node.english_name AS ETF_English_Name, score AS SearchRelevance ORDER BY SearchRelevance DESC LIMIT 5"}]

In [11]:
load_dotenv()

True

In [12]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_google_genai import ChatGoogleGenerativeAI

# LLM 모델 설정
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash-lite", temperature=0)


# Few-shot 예제 포맷팅 함수 정의
def format_few_shot_examples(examples):
    return "\n\n".join([
        f"Question: {ex['question']}\nCypher query: {ex['query']}"
        for ex in examples
    ])

# Cypher 쿼리 생성을 위한 프롬프트 템플릿 정의
CYPHER_TEMPLATE = """
당신은 질문을 Cypher로 번역하는 Neo4j 전문가입니다.
스키마: {schema}

중요사항: 
- Cypher 쿼리에서 APOC 함수를 사용하지 마세요!
- 전문 검색을 위해서는 반드시 "'etf_name_fulltext'" 인덱스 이름만 사용하세요
- 전문 검색의 올바른 형식은 다음과 같습니다:
  CALL db.index.fulltext.queryNodes("etf_name_fulltext", "검색어")
  YIELD node, score
- 펀드 이름이나 ETF 이름에서 키워드를 검색할 때는 항상 전문 검색을 사용하세요
- "RETURN *" 또는 "RETURN DISTINCT *"를 변수 없이 사용하지 마세요
- 항상 "RETURN node.name, node.code"와 같이 명시적인 반환 변수를 지정하세요
- 모든 MATCH 패턴에는 최소한 하나의 변수가 정의되어 있어야 합니다
- RETURN 문에서 사용하기 전에 항상 변수를 정의하세요
- 변수 이름은 소문자로 사용하고 이름 지정에 일관성을 유지하세요
- 전문 검색 쿼리는 반드시 다음과 같은 형식으로 작성하세요:
  CALL db.index.fulltext.queryNodes("etf_name_fulltext", "검색어")
  YIELD node, score
  RETURN node.name AS ETF_Name, node.english_name AS ETF_English_Name, score AS SearchRelevance

Few-shot Examples:
{few_shot_examples}

Question: {question}
Cypher query:
"""

CYPHER_GENERATION_PROMPT = PromptTemplate(
    input_variables=["schema", "question", "few_shot_examples"], 
    template=CYPHER_TEMPLATE
)

# Text2Cypher 체인 생성
text2cypher_chain = (
    {
        "question": lambda x: x["question"],
        "schema": lambda x: graph.schema,
        "few_shot_examples": lambda x: format_few_shot_examples(
            example_selector.select_examples({"question": x["question"]})
        )
    }
    | CYPHER_GENERATION_PROMPT 
    | llm 
    | StrOutputParser()
)

# Text2Cypher 체인 테스트
cypher_query = text2cypher_chain.invoke({"question": "전문 검색으로 AI 관련 ETF를 찾고 관련도 점수를 알려주세요."})
print(f"Cypher query: {cypher_query}")

# 쿼리 실행
result = graph.query(cypher_query)
print(result)

Cypher query: CALL db.index.fulltext.queryNodes('etf_name_fulltext', 'AI') YIELD node, score RETURN node.name AS ETF_Name, node.english_name AS ETF_English_Name, score AS SearchRelevance ORDER BY SearchRelevance DESC
[{'ETF_Name': 'TIGER 글로벌AI&로보틱스 INDXX', 'ETF_English_Name': 'TIGER Global AI & Robotics', 'SearchRelevance': 1.378050684928894}, {'ETF_Name': 'SOL 미국AI소프트웨어', 'ETF_English_Name': 'SOL US AI Software', 'SearchRelevance': 1.378050684928894}, {'ETF_Name': 'TIGER 글로벌AI인프라액티브', 'ETF_English_Name': 'TIGER GLOBAL AI INFRA ACTIVE', 'SearchRelevance': 1.2547351121902466}, {'ETF_Name': 'TIGER AI반도체핵심공정', 'ETF_English_Name': 'TIGER AI Semiconductor Core Tech', 'SearchRelevance': 1.2547351121902466}, {'ETF_Name': 'TIGER 글로벌온디바이스AI', 'ETF_English_Name': 'TIGER GLOBAL ON DEVICE AI ETF', 'SearchRelevance': 1.2547351121902466}, {'ETF_Name': 'UNICORN 생성형AI강소기업액티브', 'ETF_English_Name': 'UNICORN GEN AI SMALL GIANT ACTIVE', 'SearchRelevance': 1.151676893234253}, {'ETF_Name': 'SOL 미국AI반도체칩메이커'

In [13]:
from langchain_neo4j import GraphCypherQAChain
from langchain_google_genai import ChatGoogleGenerativeAI


# LLM 모델 설정
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash-lite", temperature=0)

# 커스텀 프롬프트 적용 GraphCypherQAChain 생성
cypher_qa_chain = GraphCypherQAChain.from_llm(
    llm=llm,
    graph=graph,
    verbose=True,
    
    allow_dangerous_requests=True,
    return_intermediate_steps=True,
    
    # few_shot_examples 처리를 위한 cypher_llm_chain_kwargs 설정
    cypher_llm_chain_kwargs={
        "llm": llm,
        "prompt": CYPHER_GENERATION_PROMPT
    }
)

# 쿼리 실행 함수
def query_graph(question):
    # 쿼리 실행
    result = cypher_qa_chain.invoke({
        "query": question,
        "few_shot_examples": format_few_shot_examples(
            example_selector.select_examples({"question": question})
        )
    })
    return result

# 예시 쿼리 테스트
question = "미래에셋자산운용에서 운용하는 ETF는 무엇인가요??"
result = query_graph(question)




> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (etf:ETF)-[:MANAGED_BY]->(am:AssetManager {name: "미래에셋자산운용"}) RETURN etf.name
Full Context:
[{'etf.name': 'TIGER AI반도체핵심공정'}, {'etf.name': 'TIGER 미국AI빅테크10'}, {'etf.name': 'TIGER 200철강소재'}, {'etf.name': 'TIGER 일본니케이225'}, {'etf.name': 'TIGER 미국나스닥100커버드콜(합성)'}, {'etf.name': 'TIGER 차이나항셍테크'}, {'etf.name': 'TIGER 미국배당다우존스'}, {'etf.name': 'TIGER 퓨처모빌리티액티브'}, {'etf.name': 'TIGER 소프트웨어'}, {'etf.name': 'TIGER S&P글로벌인프라(합성)'}]

> Finished chain.


In [14]:
# 결과 출력
for k, v in result.items():
    print(f"{k}:\n{v}")
    print("-" * 100)

query:
미래에셋자산운용에서 운용하는 ETF는 무엇인가요??
----------------------------------------------------------------------------------------------------
few_shot_examples:
Question: 인공지능 기술에 투자하는 ETF는 무엇인가요?
Cypher query: MATCH (e:ETF)-[:FOCUSES_ON]->(t:Technology {name: '인공지능'}) RETURN e.korean_name, e.code, e.basic_info LIMIT 5

Question: 교보악사자산운용에서 관리하는 ETF는 무엇인가요?
Cypher query: MATCH (e:ETF)-[:MANAGED_BY]->(:AssetManager {name: '교보악사자산운용'}) RETURN e.korean_name, e.code, e.fund_type LIMIT 5

Question: 인버스 ETF 중 거래량이 가장 많은 것은?
Cypher query: MATCH (e:ETF) WHERE e.tracking_multiplier < 0 RETURN e.korean_name, e.code, e.trading_volume ORDER BY e.trading_volume DESC LIMIT 5
----------------------------------------------------------------------------------------------------
result:
미래에셋자산운용에서 운용하는 ETF는 TIGER AI반도체핵심공정, TIGER 미국AI빅테크10, TIGER 200철강소재, TIGER 일본니케이225, TIGER 미국나스닥100커버드콜(합성), TIGER 차이나항셍테크, TIGER 미국배당다우존스, TIGER 퓨처모빌리티액티브, TIGER 소프트웨어, TIGER S&P글로벌인프라(합성)입니다.
-------------------------------

In [ ]:
from langchain_neo4j import GraphCypherQAChain
from langchain_core.prompts import ChatPromptTemplate

# ETF 추천을 위한 프롬프트 템플릿 정의
ETF_RECOMMENDATION_TEMPLATE = """
당신은 ETF 데이터베이스 분석 전문가로서 ETF 데이터에 대한 명확하고 간결한 정보를 한국어로 제공합니다.

질문: {question}
검색 결과: {context}

응답 가이드라인:
- 검색 결과에서 핵심 정보를 요약하세요
- ETF 데이터에 대한 명확하고 객관적인 개요를 제공하세요
- 전문적이고 유익한 톤을 사용하세요
- ETF 데이터의 중요한 패턴이나 추세를 강조하세요
- 컨텍스트가 부족하거나 근거가 없을 경우 정보가 부족하다고 명확히 언급하세요
- 추측이나 개인적인 해석은 피하세요
- 데이터베이스에서 관련 정보를 찾을 수 없는 경우, 정확한 정보를 제공할 수 없다고 명시하세요

응답 형식:
- 간략한 발견 요약으로 시작하세요
- 여러 ETF가 발견된 경우 간결한 개요를 제공하세요
- 가독성을 위해 글머리 기호나 짧은 단락을 사용하세요
- 상장일, 수수료율, 투자 테마, 시장 등과 같은 관련 세부 정보를 포함하세요
- 수치 데이터나 기술 용어를 쉽게 이해할 수 있는 언어로 번역하세요

응답 구조 예시:
"분석 결과: [주요 발견 사항 요약]

주요 특징:
- [첫 번째 중요한 통찰]
- [두 번째 중요한 통찰]

추가 정보: [필요한 경우 추가 설명]"
"""

# ETF 추천 프롬프트 템플릿 생성
etf_recommendation_prompt = ChatPromptTemplate.from_template(ETF_RECOMMENDATION_TEMPLATE)


# ETF 추천을 위한 GraphCypherQAChain 생성
#cypher_prompt가 '쿼리 생성'을 담당한다면, qa_prompt는 '결과 해석'을 담당

etf_qa_chain = GraphCypherQAChain.from_llm(
    llm=llm,
    graph=graph,  
    verbose=True,
    cypher_prompt=CYPHER_GENERATION_PROMPT,
    qa_prompt=etf_recommendation_prompt,
    validate_cypher=True,
    return_intermediate_steps=True,
    allow_dangerous_requests=True,
    top_k=5,
    return_direct=False,
    
    # few_shot_examples 처리를 위한 cypher_llm_chain_kwargs 설정
    cypher_llm_chain_kwargs={
        "llm": llm,
        "prompt": CYPHER_GENERATION_PROMPT
    }
)

# 쿼리 실행 함수
def query_graph(question):
    # 쿼리 실행
    result = etf_qa_chain.invoke({
        "query": question,
        "few_shot_examples": format_few_shot_examples(
            example_selector.select_examples({"question": question})
        )
    })
    return result

# 예시 쿼리 테스트
question = "전문 검색으로 AI 관련 ETF를 찾고 관련도 점수를 알려주세요."
result = query_graph(question)



> Entering new GraphCypherQAChain chain...
Generated Cypher:
CALL db.index.fulltext.queryNodes('etf_name_fulltext', 'AI') YIELD node, score RETURN node.name AS ETF_Name, node.english_name AS ETF_English_Name, score AS SearchRelevance ORDER BY SearchRelevance DESC
Full Context:
[{'ETF_Name': 'TIGER 글로벌AI&로보틱스 INDXX', 'ETF_English_Name': 'TIGER Global AI & Robotics', 'SearchRelevance': 1.378050684928894}, {'ETF_Name': 'SOL 미국AI소프트웨어', 'ETF_English_Name': 'SOL US AI Software', 'SearchRelevance': 1.378050684928894}, {'ETF_Name': 'TIGER 글로벌AI인프라액티브', 'ETF_English_Name': 'TIGER GLOBAL AI INFRA ACTIVE', 'SearchRelevance': 1.2547351121902466}, {'ETF_Name': 'TIGER AI반도체핵심공정', 'ETF_English_Name': 'TIGER AI Semiconductor Core Tech', 'SearchRelevance': 1.2547351121902466}, {'ETF_Name': 'TIGER 글로벌온디바이스AI', 'ETF_English_Name': 'TIGER GLOBAL ON DEVICE AI ETF', 'SearchRelevance': 1.2547351121902466}]

> Finished chain.


In [16]:
print(result['result'])

분석 결과: AI 관련 ETF 검색 결과, 총 5개의 ETF가 관련성이 높은 것으로 나타났습니다. 이 ETF들은 AI 기술의 다양한 측면에 투자하며, 검색 관련도 점수는 1.25에서 1.38 사이로 나타났습니다.

주요 특징:

*   **높은 관련도:** 'TIGER 글로벌AI&로보틱스 INDXX'와 'SOL 미국AI소프트웨어'가 1.378의 가장 높은 검색 관련도 점수를 기록하며 AI 분야에 대한 높은 연관성을 보여줍니다.
*   **다양한 투자 테마:** AI 관련 ETF들은 다음과 같은 세부 투자 테마를 포함하고 있습니다.
    *   글로벌 AI 및 로보틱스 (TIGER 글로벌AI&로보틱스 INDXX)
    *   미국 AI 소프트웨어 (SOL 미국AI소프트웨어)
    *   글로벌 AI 인프라 (TIGER 글로벌AI인프라액티브)
    *   AI 반도체 핵심 공정 (TIGER AI반도체핵심공정)
    *   글로벌 온디바이스 AI (TIGER 글로벌온디바이스AI)
*   **일관된 관련도 점수:** 상위 2개 ETF를 제외한 나머지 ETF들도 1.254의 유사한 관련도 점수를 보여, 전반적으로 AI 분야에 집중된 ETF들이 검색 결과에 포함되었음을 알 수 있습니다.

추가 정보:

제공된 데이터에는 각 ETF의 상장일, 수수료율, 구체적인 투자 종목 등 상세 정보가 포함되어 있지 않습니다. 따라서 각 ETF의 투자 전략, 성과, 위험도 등을 더 깊이 이해하기 위해서는 추가적인 정보 조사가 필요합니다.


In [17]:
print(result['intermediate_steps'])

[{'query': "CALL db.index.fulltext.queryNodes('etf_name_fulltext', 'AI') YIELD node, score RETURN node.name AS ETF_Name, node.english_name AS ETF_English_Name, score AS SearchRelevance ORDER BY SearchRelevance DESC"}, {'context': [{'ETF_Name': 'TIGER 글로벌AI&로보틱스 INDXX', 'ETF_English_Name': 'TIGER Global AI & Robotics', 'SearchRelevance': 1.378050684928894}, {'ETF_Name': 'SOL 미국AI소프트웨어', 'ETF_English_Name': 'SOL US AI Software', 'SearchRelevance': 1.378050684928894}, {'ETF_Name': 'TIGER 글로벌AI인프라액티브', 'ETF_English_Name': 'TIGER GLOBAL AI INFRA ACTIVE', 'SearchRelevance': 1.2547351121902466}, {'ETF_Name': 'TIGER AI반도체핵심공정', 'ETF_English_Name': 'TIGER AI Semiconductor Core Tech', 'SearchRelevance': 1.2547351121902466}, {'ETF_Name': 'TIGER 글로벌온디바이스AI', 'ETF_English_Name': 'TIGER GLOBAL ON DEVICE AI ETF', 'SearchRelevance': 1.2547351121902466}]}]
